# IF29 — Analyse exploratoire des données (EDA)

## Projet : Détection de profils atypiques sur X (Twitter)

### Logique du projet (ordre strict)

```
1. Agrégation MongoDB
      ↓
   users_aggregated.csv        ← 643 124 profils · 21 variables · SANS label
      ↓
2. EDA (CE NOTEBOOK)          ← describe, distributions, corrélations, outliers
      ↓
   Justification : 21 → 16 features ML
      ↓
3. Labellisation Excel        ← Groupe3_Labelisation.ipynb
      ↓
   users_labeled_manual.csv   ← même base + colonnes label + anomaly_score
      ↓
4. Modélisation ML            ← notebooks non supervisé / supervisé
```

**Règle importante :** l'EDA ne utilise **que** `users_aggregated.csv`.  
On ne labellise **qu'après** avoir compris les données et retenu les 16 variables.

**Livrable de ce notebook :** la justification des **16 features** retenues pour le ML.


## 1. Cadre analytique

> *« Un profil atypique est un profil dont les caractéristiques statistiques ou comportementales s'écartent significativement de la population moyenne. »*

Dimensions étudiées (Ferrara et al., 2016 ; Chu et al., 2012 ; Varol et al., 2017 ; SPOT) :

| Dimension | Indicateurs | Signal d'anomalie |
|-----------|-------------|-------------------|
| Graphe social | followers, friends, ratio | Ratio extrême |
| Activité | nb_tweets, retweet_ratio | Amplification / burst |
| Contenu | hashtags, URLs, mentions | Spam / surcharge |
| Temporel | active_days, tweet_frequency | Activité concentrée |
| Visibilité (SPOT) | followers × retweet_ratio | Portée d'amplification |

**Note :** la définition opérationnelle des labels (4 critères Excel) est appliquée **après** l'EDA — `docs/LABELISATION.md`.


In [ ]:
import os
import glob
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

import os
import glob
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

RAW_DIR = "raw"
DATA_AGG = "users_aggregated.csv"   # SEUL fichier utilisé dans ce notebook (EDA)
RANDOM_STATE = 42

VARIABLES_AGGREGATED_COUNT = 21
FEATURES_ML_COUNT = 16

FEATURES_EXCLUDED_FROM_ML = [
    "user_id", "screen_name", "profile_lang",
    "first_tweet_date", "last_tweet_date",
]

FEATURES_ML = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "nb_retweets", "retweet_ratio",
    "avg_tweet_length", "avg_hashtags", "avg_urls", "avg_mentions",
    "avg_favorites", "avg_retweet_count",
    "active_days", "tweet_frequency", "verified", "default_profile_image",
]

print("EDA — source :", DATA_AGG)
print(f"Agrégation : {VARIABLES_AGGREGATED_COUNT} variables → ML : {FEATURES_ML_COUNT} features")
print("Labellisation (étape suivante) → users_labeled_manual.csv")


## 2. Données brutes — vue d'ensemble


In [ ]:
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.json")))
print(f"Nombre de fichiers JSONL : {len(raw_files)}")

sample_tweets = []
with open(raw_files[0], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        sample_tweets.append(json.loads(line))

tweet = sample_tweets[0]
print(f"Structure tweet : {list(tweet.keys())[:10]}")
print(f"Auteur (user) : {list(tweet['user'].keys())[:8]}")
print(f"Est un retweet : {'retweeted_status' in tweet}")

pd.DataFrame([{
    "user_id": t["user"]["id"],
    "screen_name": t["user"]["screen_name"],
    "text_len": len(t.get("text", "")),
    "is_retweet": "retweeted_status" in t,
} for t in sample_tweets])


**Pipeline Étape 1 :** tweets bruts -> enrichissement -> agrégation MongoDB par `user.id` -> `users_aggregated.csv`

**Hypothèse fondamentale :** seul l'auteur du tweet (`user`) est analysé, pas `retweeted_status.user`.


## 3. Chargement et aperçu du dataset agrégé


In [ ]:
df = pd.read_csv(DATA_AGG)

print(f"Dataset chargé : {df.shape[0]:,} utilisateurs, {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")
df.head()


In [ ]:
print("=" * 60)
print(f"STATISTIQUES DESCRIPTIVES — {len(df):,} profils (jeu complet)")
print("=" * 60)

df_eda = df.copy()
df_eda["verified"] = df_eda["verified"].astype(int)
df_eda["default_profile_image"] = df_eda["default_profile_image"].astype(int)

num_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()
describe_full = df_eda[num_cols].describe().T.round(2)
display(describe_full)

print(f"\n{len(num_cols)} colonnes numériques sur {df.shape[1]} colonnes agrégées.")


## 4. Valeurs manquantes et types


In [ ]:
print("=" * 60)
print("TYPES ET VALEURS MANQUANTES")
print("=" * 60)
df.info()

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Valeurs manquantes": missing,
    "Pourcentage (%)": missing_pct
}).sort_values("Pourcentage (%)", ascending=False)

print("\nValeurs manquantes par colonne :")
display(missing_df[missing_df["Valeurs manquantes"] > 0])

if missing_df["Valeurs manquantes"].sum() == 0:
    print("\nAucune valeur manquante détectée.")
else:
    missing_pct[missing_pct > 0].plot(kind="bar", color="coral", edgecolor="black")
    plt.title("Pourcentage de valeurs manquantes par colonne")
    plt.ylabel("% manquant")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 5. Distribution des variables principales

Sur Twitter, la plupart des utilisateurs ont peu de followers, mais quelques-uns en ont énormément.
Distribution asymétrique (loi de puissance), classique sur les réseaux sociaux.
On utilise `log1p(x) = log(1+x)` pour mieux visualiser.


In [ ]:
cols_to_plot = [
    "followers_count", "friends_count", "nb_tweets", "nb_retweets",
    "retweet_ratio", "avg_hashtags", "avg_mentions", "followers_friends_ratio"
]
cols_to_plot = [c for c in cols_to_plot if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    data = np.log1p(df[col].clip(lower=0).dropna())
    axes[i].hist(data, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{col}\n(échelle log)", fontsize=10)
    axes[i].set_xlabel("log(1 + valeur)")

for j in range(len(cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution des variables principales (échelle logarithmique)", fontweight="bold")
plt.tight_layout()
plt.show()

print("INTERPRETATION :")
print("- Distributions tres asymetriques (queue longue a droite)")
print("- Quelques comptes ont des valeurs enormes (celebrites / bots)")
print("- log1p ici pour la visualisation ; StandardScaler seul avant ML (section 12)")


In [ ]:
box_cols = ["followers_count", "friends_count", "nb_tweets", "retweet_ratio", "avg_hashtags", "active_days"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, col in zip(axes, box_cols):
    data = df[col].clip(upper=df[col].quantile(0.95))
    ax.boxplot(data, vert=True)
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
plt.suptitle("Boxplots (95e percentile) — presence d'outliers", fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Sélection des variables — 21 → 16 features ML

**Livrable principal de l'EDA.** L'agrégation MongoDB produit **21 colonnes** ; nous en retenons **16** pour la modélisation :


In [ ]:
selection_df = pd.DataFrame({
    "Variable exclue": FEATURES_EXCLUDED_FROM_ML,
    "Raison": [
        "Identifiant numérique — non prédictif",
        "Identifiant texte — non prédictif",
        "Catégorielle peu informative pour le ML",
        "Redondante avec active_days / tweet_frequency",
        "Redondante avec active_days / tweet_frequency",
    ],
})
display(selection_df)

print(f"\nVariables agrégées : {VARIABLES_AGGREGATED_COUNT}")
print(f"Features ML retenues : {FEATURES_ML_COUNT}")
print("Liste :", FEATURES_ML)

assert len(FEATURES_ML) == FEATURES_ML_COUNT
assert set(FEATURES_ML).issubset(set(df.columns))


## 7. Matrice de corrélation (%) — 16 features ML

Corrélations de Pearson sur les **16 features retenues**, exprimées en **pourcentage** (coefficient × 100).

In [ ]:
df_ml = df_eda[FEATURES_ML].astype(float)
corr_matrix = df_ml.corr()
corr_pct = (corr_matrix * 100).round(1)

display(corr_pct)

plt.figure(figsize=(14, 11))
annot = corr_pct.applymap(lambda x: f"{x:.0f}%")
sns.heatmap(
    corr_pct, annot=annot, fmt="",
    cmap="RdBu_r", center=0, vmin=-100, vmax=100,
    linewidths=0.4, square=True,
    cbar_kws={"label": "Corrélation (%)"},
    annot_kws={"size": 7},
)
plt.title("Matrice de corrélation (%) — 16 features ML", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nPaires fortement corrélées (|r| > 50 %) :")
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r_pct = corr_pct.iloc[i, j]
        if abs(r_pct) > 50:
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]} : {r_pct:.1f} %")

## 8. Feature engineering (exploratoire)

Variables dérivées pour explorer les comportements suspects (Chu et al., 2012 ; Ferrara et al., 2016 ; SPOT).

**Note :** ces features engineered **ne font pas partie des 16 features ML** retenues pour la modélisation. `avg_favorites` et `avg_retweet_count` sont nulles (collecte partielle).


In [ ]:
df_features = df.copy()

df_features["activity_rate"] = (
    df_features["nb_tweets"] / (df_features["active_days"] + 1)
).clip(0, 1)

df_features["engagement_score"] = (
    df_features["avg_favorites"] + df_features["avg_retweet_count"]
)

df_features["aggressiveness_score"] = (
    df_features["activity_rate"] * np.log1p(df_features["friends_count"])
)

df_features["visibility_score"] = (
    np.log1p(df_features["followers_count"]) * (1 + df_features["retweet_ratio"])
)

df_features["amplification_index"] = (
    df_features["nb_retweets"] / (df_features["avg_retweet_count"] + 1)
)

df_features["content_richness"] = (
    df_features["avg_hashtags"] + df_features["avg_urls"] + df_features["avg_mentions"]
)

df_features["is_verified"] = df_features["verified"].astype(int)

engineered = [
    "activity_rate", "aggressiveness_score", "visibility_score",
    "amplification_index", "content_richness", "engagement_score"
]
print("Features engineered calculées :")
for f in engineered:
    print(f"  {f:25s}  médiane={df_features[f].median():.3f}  max={df_features[f].max():.3f}")


In [ ]:
features_to_viz = [
    "followers_friends_ratio", "retweet_ratio", "activity_rate",
    "aggressiveness_score", "visibility_score", "amplification_index", "content_richness"
]
features_to_viz = [f for f in features_to_viz if f in df_features.columns]

ncols = 3
nrows = (len(features_to_viz) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for i, feat in enumerate(features_to_viz):
    data = df_features[feat].replace([np.inf, -np.inf], np.nan).dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    axes[i].hist(data.clip(p1, p99), bins=50, color="teal", edgecolor="white", alpha=0.8)
    axes[i].set_title(feat, fontsize=11, fontweight="bold")

for j in range(len(features_to_viz), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution des features engineered (tronquées P1-P99)", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
sample = df_features[["aggressiveness_score", "visibility_score"]].dropna()
sample = sample[np.isfinite(sample).all(axis=1)]
sample = sample.sample(min(5000, len(sample)), random_state=RANDOM_STATE)

plt.figure(figsize=(10, 7))
plt.scatter(sample["aggressiveness_score"], sample["visibility_score"],
            alpha=0.3, s=10, c="steelblue")
plt.xlabel("Aggressiveness Score (adapté SPOT)")
plt.ylabel("Visibility Score (adapté SPOT)")
plt.title("Scatter : Agressivité vs Visibilité", fontweight="bold")

q90_agg = sample["aggressiveness_score"].quantile(0.9)
q90_vis = sample["visibility_score"].quantile(0.9)
plt.axhline(q90_vis, color="red", linestyle="--", alpha=0.6, label="90e percentile visibilité")
plt.axvline(q90_agg, color="orange", linestyle="--", alpha=0.6, label="90e percentile agressivité")
plt.legend()
plt.tight_layout()
plt.show()

n_suspects = ((sample["aggressiveness_score"] > q90_agg) &
              (sample["visibility_score"] > q90_vis)).sum()
print(f"Profils quadrant suspect (agg. ET vis. > P90) : {n_suspects:,} ({n_suspects/len(sample)*100:.2f} %)")


## 9. Profils extrêmes — méthode IQR


In [ ]:
def count_outliers(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_cols = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "retweet_ratio", "aggressiveness_score", "visibility_score"
]
outlier_stats = pd.DataFrame({
    "Outliers (IQR)": [count_outliers(df[c]) for c in outlier_cols],
    "Pct (%)": [count_outliers(df[c]) / len(df) * 100 for c in outlier_cols],
    "P99": [df[c].quantile(0.99) for c in outlier_cols],
    "Médiane": [df[c].median() for c in outlier_cols],
}, index=outlier_cols)
display(outlier_stats.round(2))

print("\nTop 5 — ratio followers/friends le plus élevé :")
display(df.nlargest(5, "followers_friends_ratio")[
    ["screen_name", "followers_count", "friends_count", "followers_friends_ratio", "nb_tweets", "retweet_ratio"]
])


In [ ]:
sample = df.sample(n=5000, random_state=RANDOM_STATE)
plt.figure(figsize=(10, 6))
plt.scatter(
    sample["followers_count"].clip(upper=sample["followers_count"].quantile(0.99)),
    sample["nb_tweets"], alpha=0.3, s=10, c="steelblue"
)
plt.xlabel("followers_count (99e pct)")
plt.ylabel("nb_tweets")
plt.title("Followers vs Activité (n=5 000)", fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. ACP — réduction dimensionnelle (16 features, sans labels)

ACP sur `users_aggregated.csv` — **avant labellisation**. Choix retenu pour le non supervisé : **7 composantes** (seuil 75 % de variance).


## 11. Normalisation (aperçu)

Même prétraitement que les notebooks ML — sur les **16 features** de `users_aggregated.csv` :

- **`StandardScaler`** : z = (x − mean) / std
- `log1p` réservé à la **visualisation** uniquement


In [ ]:
X_full = StandardScaler().fit_transform(df_eda[FEATURES_ML].astype(float))
pca_full = PCA(n_components=10, random_state=RANDOM_STATE).fit(X_full)
var_cum = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(var_cum) + 1), var_cum, "o-", color="steelblue")
ax.axhline(75, color="red", linestyle="--", label="Seuil 75 %")
ax.axvline(7, color="orange", linestyle=":", linewidth=2, label="Retenu = 7 comp.")
ax.set_xlabel("Nombre de composantes")
ax.set_ylabel("Variance cumulée (%)")
ax.set_title("ACP — Variance cumulée (16 features ML, users_aggregated.csv)", fontweight="bold")
ax.legend()
ax.grid(alpha=0.35)
plt.tight_layout()
plt.show()

print("Variance cumulée ACP :")
for i, v in enumerate(var_cum[:7], 1):
    print(f"  {i} composante(s) -> {v:.1f} %")
print(f"\n=> 7 composantes retenues pour le non supervisé (~{var_cum[6]:.1f} %)")


In [ ]:
X = df_eda[FEATURES_ML].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_ML)

print("Données normalisées — Shape :", X_scaled.shape)
print("\nVérification (mean ≈ 0, std ≈ 1) :")
X_scaled_df.describe().T[["mean", "std"]].round(3)


## 12. Prochaine étape — Labellisation (hors EDA)

Une fois les **16 features** retenues, l'équipe a réalisé une **labellisation manuelle sous Excel** pour créer le fichier utilisé par le ML :

```
users_aggregated.csv  +  règles Excel  →  users_labeled_manual.csv
   (21 variables)                              (+ label, + anomaly_score)
```

| Fichier | Rôle | Labels |
|---------|------|--------|
| `users_aggregated.csv` | Sortie agrégation + **EDA** | Non |
| `users_labeled_manual.csv` | Entrée **ML** (non sup. + sup.) | Oui |

**4 critères Excel** — atypique si ≥ 2 sur 4. Détail : `Groupe3_Labelisation.ipynb` · `docs/LABELISATION.md`

> Ce notebook s'arrête ici. La labellisation n'appartient pas à l'EDA.


In [ ]:
print("=" * 60)
print("FIN DE L'EDA — users_aggregated.csv")
print("=" * 60)
print(f"Variables agrégées MongoDB : {VARIABLES_AGGREGATED_COUNT}")
print(f"Features ML retenues      : {FEATURES_ML_COUNT}")
print(f"Exclues après EDA         : {FEATURES_EXCLUDED_FROM_ML}")
print("\n→ Étape suivante : labellisation Excel → users_labeled_manual.csv")


## 12. Tableau récapitulatif — 16 features ML retenues


In [ ]:
recap = pd.DataFrame([
    {"Feature": f, "Source": "MongoDB", "Retenue après EDA": "Oui"}
    for f in FEATURES_ML
])
recap_excl = pd.DataFrame([
    {"Variable exclue": v, "Raison": r}
    for v, r in zip(
        FEATURES_EXCLUDED_FROM_ML,
        ["Identifiant", "Identifiant", "Catégorielle", "Date brute", "Date brute"],
    )
])

print("=" * 60)
print(f"FEATURES ML RETENUES ({FEATURES_ML_COUNT})")
print("=" * 60)
display(recap)

print("\nVARIABLES EXCLUES APRÈS EDA :")
display(recap_excl)


## 13. Synthèse et prochaines étapes

**Observations clés (EDA sur `users_aggregated.csv`) :**
1. **643 124 profils** — distributions asymétriques (loi de puissance)
2. Peu de valeurs manquantes · `describe` sur le jeu complet
3. **21 variables agrégées → 16 features ML** (5 exclues après EDA)
4. Corrélations fortes entre certaines variables → ACP en modélisation
5. **ACP : 7 composantes ~ 79 %** variance (non supervisé)

**Étape suivante :** `Groupe3_Labelisation.ipynb` → `users_labeled_manual.csv`

| Notebook | Rôle |
|----------|------|
| `Groupe3_Labelisation.ipynb` | Labellisation Excel |
| `Groupe3_profils_atypiques_non_Sup.ipynb` | K-Means / Isolation Forest |
| `Groupe3_profils_atypiques_Sup.ipynb` | SVM / XGBoost |
| `Groupe3_profils_atypiques_Final.ipynb` | Synthèse |


In [ ]:
recap_eda = pd.DataFrame({
    "Métrique": [
        "Fichier EDA",
        "Profils agrégés",
        "Variables agrégées MongoDB",
        "Features ML retenues",
        "Comptes vérifiés",
        "Médiane followers",
        "Médiane nb_tweets",
        "Variance ACP (7 comp.)",
        "Quadrant suspect SPOT (P90)",
    ],
    "Valeur": [
        DATA_AGG,
        f"{len(df):,}",
        str(VARIABLES_AGGREGATED_COUNT),
        str(FEATURES_ML_COUNT),
        f"{df['verified'].mean()*100:.2f} %",
        f"{df['followers_count'].median():,.0f}",
        f"{df['nb_tweets'].median():,.0f}",
        f"{var_cum[6]:.1f} %",
        f"{n_suspects:,} ({n_suspects/len(sample)*100:.2f} %)",
    ],
})
print("=" * 50)
print(" RECAPITULATIF EDA")
print("=" * 50)
display(recap_eda)


In [ ]:
# Fin du notebook EDA
